<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="assets/content/images/tensorflow_thumbnail-01.png" align="center" width="20%">
</div>

<br>

# AN INTRODUCTION TO TENSORFLOW V.2

<br>

**About:** TensorFlow (TF) is an open-source library for dataflow programming, differentiable computation, and machine learning. This notebook introduces TF V.2 from the ground up so that readers without a prior machine learning background can build confidence with the framework before applying it to real models.

**Learning Goals:**
- Create and manipulate `tf.constant` and `tf.Variable` tensors across scalar, vector, matrix, and higher-rank shapes
- Perform arithmetic, linear algebra, gradient, and slicing operations on tensors
- Understand eager execution and why it simplifies debugging compared to TF 1.x graph execution
- Set up TensorBoard and interpret its training curves
- Decorate Python functions with `@tf.function` to compile them into portable TensorFlow graphs
- Recognize when to use `@tf.function` and when to stay in eager mode
- Handle special tensor types: ragged, string, and sparse

**Keywords:** tensorflow, tensorflow-tutorial, deep-learning, deep-learning-with-python, neural-networks, keras

**Prerequisite Knowledge:** (1) Python, (2) NumPy, (3) Pandas, (4) Linear Algebra

**Target User:** Data scientists, applied machine learning engineers, and developers

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: ABOUT AND MOTIVATION](#Part_0)
> #### [PART 1: TENSORS AND OPERATIONS](#Part_1)
> #### [PART 2: TENSORFLOW GRAPHS AND EXECUTIONS](#Part_2)
> #### [PART 3: WRAP UP AND NEXT STEPS](#Part_3)

#### APPENDIX

> #### [TENSORFLOW INSTALLATION](#Appendix_1)
> #### [REFERENCES AND ADDITIONAL RESOURCES](#Appendix_2)

<br>

#### APPENDIX

> #### [TENSORFLOW INSTALLATION](#Appendix_1)
> #### [REFERENCES AND ADDITIONAL RESOURCES](#Appendix_2)

<br>


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **ABOUT** AND **MOTIVATION**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tensorflow_thumbnail_play-01.png" align="center" width="50%" padding="10px"/><br>
</div>

TensorFlow V.2 is Google's open-source library for numerical computation and differentiable programming. It is the foundation for a large share of production machine learning systems at scale, and it ships with Keras as its high-level API. Understanding TensorFlow at the tensor and graph level - before reaching for `model.fit()` - gives you two lasting advantages:

1. **Debuggability.** Knowing how eager execution works, and what `@tf.function` compiles into, lets you reason about why a model trains slowly or produces unexpected gradients - rather than treating the framework as a black box.
2. **Portability.** A TensorFlow graph is hardware-agnostic: the same decorated function runs identically on a laptop CPU, a cloud GPU, and a mobile TPU delegate. Understanding the graph is the precondition for deploying models beyond a Jupyter notebook.

This notebook builds that foundation in two phases. **Part 1** covers the core data structure (tensors) and the operations that act on them, including automatic differentiation via `tf.GradientTape`. **Part 2** covers the computation graph model: how `@tf.function` traces and compiles Python functions, what tracing means for side effects, and when to prefer graph execution over eager mode.

___

**Note:** This notebook uses TensorFlow V.2 exclusively. TF 1.x code used explicit `Session.run()` calls and required building the full graph before any value could be inspected. V.2 makes the session invisible - eager execution is the default - so you can treat tensor operations much like NumPy operations during development.

___

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **TENSORS** AND **OPERATIONS**


<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tensorflow_thumbnail-01.png" align="center" width="40%" padding="10"><br>
    <br>
    
</div>

<br>

<br>

[**TensorFlow**](https://www.tensorflow.org/) is a cross-platform, end-to-end, open source, platform for efficiently training and deploying machine learning models. TensorFlow offers multiple levels of abstractions, which means you can use high-level API's such as [**Keras**](https://keras.io/) or [**Ludwig**](https://ludwig-ai.github.io/ludwig-docs/index.html) to make things a bit simpler, or even to set up a [**Distribution Strategy**](https://www.tensorflow.org/guide/distributed_training) API on different hardware configurions without having to change the model defintion.

<br>

<strong style="color:red">KEY CONSIDERATION:</strong> Some of the following content may be written for machines running on Linux or Mac operating systems. If you are working on a Windows machine, you will need to enable the Linux Bash Shell, or adjust Shell commands to PowerShell syntax. A tutorial on how to enable the Linux Bash Shell on Windows 10 can be found [here](https://youtu.be/xzgwDbe7foQ).


#### CONTENTS:

> [PART 1.1: TENSORFLOW SETUP](#Part_1_1)<br>
> [PART 1.2: TENSORBOARD SETUP](#Part_1_2)<br>
> [PART 1.3: TENSORFLOW TENSORS](#Part_1_3)<br>
> [PART 1.4: TENSORFLOW OPERATIONS](#Part_1_4)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: TENSORFLOW SETUP

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tf_logo_social.png" align="center" width="30%" padding="0px"><br>
    <br>
</div>

<br>

<br>

[**TensorFlow V.2**](https://www.tensorflow.org/tutorials/quickstart/beginner) removes redundant APIs, and integrates more smoothly with Python via Eager execution. To learn what has changed between versions 1 and 2 of TensorFlow, see [here](https://www.tensorflow.org/guide/effective_tf2). 
<br>

<br> 

#### **1.1.1 General Notebook Setup**
___

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

<br>

**Install TensorFlow:** the following will install TensorFlow using bash. It serves our purposes. However, for detailed installation instructions see [Appendix I](#Appendix_1)

In [ ]:
## Create Virtual Environment ##
! python3 -m venv ./assets/venv

In [ ]:
## Activate Virtual Environment ##
! . ./assets/venv/bin/activate

In [ ]:
## Ensure pip version >= 19.0 ##
! pip install --upgrade pip 

In [ ]:
## Ensure TF version >= 2.0 ##
! pip install --upgrade tensorflow

<br>

**Import TensorFlow**

In [ ]:
# Canonical way of importing TensorFlow
import tensorflow as tf

___

**Note:** If ```import tensorflow as tf``` doesn't work, TensorFlow is not installed correctly. To resolve the issue see [**build and install error messages**](https://www.tensorflow.org/install/errors)
    
___

In [ ]:
# Check tf version, oftentimes tensorflow is not backwards compatible
tf.__version__

In [ ]:
# Check thatEager Execution is active
tf.executing_eagerly()

___

**Note:** TensorFlow V.2 comes with eager execution enabled by default. Eager mode has many benefits, but for our purposes eager mode allows tensor outputs to be viewed without the need of a session - which may not be necessary depending on your needs. To learn more check out the section on [**eager execution**](#Part_1_5).
___

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: TENSORBOARD SETUP

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tensorboard_logo_social.png" align="center" width="30%" padding="0px"><br>
    <br>
</div>

<br>

[**TensorBoard**](https://www.tensorflow.org/tensorboard/get_started) is a tool for measuring and visualizing machine learning workflows. It enables the tracking of important metrics such as [**acuracy**](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/Accuracy), it can display summary operations, tensor ouputs, project embeddings to lower dimensional space, and much more. This broad functionality makes TensorBoard an important tool in evaluating, optimizing, and debugging machine learning models. For a five-minute summary of TensorBoard, click [here](https://youtu.be/3bownM3L5zM).

<br>

#### **1.2.1 Load TensorBoard and Associated Libraries**

___

**Load TensorBoard**

In [ ]:
## Load TensorBoard notebook extension ##
%load_ext tensorboard

In [ ]:
## Load additional libraries needed ##
from datetime import datetime
from IPython.display import clear_output, Image, display, HTML
import os
import pathlib

t = datetime.utcnow().strftime("%Y%m%d%H%M%S") 
log_dir = "tf_logs"
logd = "/tmp/{}/r{}/".format(log_dir, t)

**Clear TensorBoard Logs**

In [ ]:
## Clear any logs from previous runs ##
!rm -rf ./logs/

<br>

#### **1.2.2 Confirm TensorBoard is running**

**Note:** The purpose of this section is only to highlight some of the commands for using TensorBoard. The [**MNIST**](https://en.wikipedia.org/wiki/MNIST_database) data set used here, is only for the purposes of highlighting functionality which on this case depends on the [**Keras**](https://keras.io/) library.

___

In [ ]:
# from Keras datasets import mnist
mnist = tf.keras.datasets.mnist

# load data into training and testing splits
(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# create a sequential model
def create_model():
  return tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
  ])

___

**Note:** Click on the following to learn more about the command:
>[```keras.model.sequential( )```](https://keras.io/api/models/sequential/)<br>
>[```keras.layers.Flatten( )```](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)<br>
>[```keras.layers.Dense( ) ```](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)<br>
>[```keras.layers.Dropout( )```](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout)
    
___

**Add Tensorboard callbacks**

In [ ]:
# instantiate model class
model = create_model()

# compile model 
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# place logs in a timestamped subdirectory and enable histogram computation with every epoch
log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)


# Fit model
model.fit(x=x_train, 
          y=y_train, 
          epochs=3, 
          validation_data=(x_test, y_test), 
          # Adding 'callbacks=[tensorboard_callback]' to model.fit( ) ensures that logs are created and stored
          callbacks=[tensorboard_callback])

___

**Note:** Click on the following to learn more about the command:
>[```model.compile( )```](https://keras.io/api/models/model/)<br>
>[```callbacks.TensorBoard( )```](https://keras.io/api/callbacks/)<br>
>[```model.fit( ) ```](https://keras.io/api/models/model_training_apis/)
    
___

**Display Tensorboard Logs**

In [ ]:
## %tensorboard line magic. In command line run same command without "%" ##
%tensorboard --logdir logs/fit

___
**Note:** if TensorBoard does not load after following the above command, try reloading TensordBoard ```%reload_ext tensorboard``` 
___

<a id='Part_1_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.3: TENSORFLOW TENSORS

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-03.png" align="center" width="50%" padding="0px"><br>
</div>

[**Tensors**](https://www.tensorflow.org/api_docs/python/tf/Tensor) are immutable multidimensional arrays for working with data with more than 2-Dimensions. Tensors essentially multilinear maps from vector spaces to real numbers. Hence a tensor can be used to represent scalars, vectors, and matrices. Moreover, tensors are [**highly efficient**](https://realpython.com/numpy-tensorflow-performance/), can be ran on several architectures (CPUs, GPUs, Mobile, and Distributed), and make the calculation of gradients easier - this is particulary useful as the analytical solution of gradients are extremely tedious to derive. To view the complete list of TensorFlow data types, see [here](https://www.tensorflow.org/api_docs/python/tf/dtypes/DType).

<br>

#### **1.3.1 tf.constant**

___

<br>

[**Constants**](https://www.tensorflow.org/api_docs/python/tf/constant) are tensors, are initialized directly, and are immutable once created. In order to evaluate them, we have to run them in a [**session**](https://www.tensorflow.org/api_docs/java/reference/org/tensorflow/Session), shown below, or can be displayed using [**eager execution**](https://www.tensorflow.org/guide/eager) without a session.

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-12.png" align="center" width="40%" padding="0px"><br>
    <br>
    Scalar (Rank-0) Tensor
</div>

In [ ]:
# int32 tensors by default
rank_0_tensor = tf.constant(4)
b = tf.constant(5)

In [ ]:
rank_0_tensor.numpy()

In [ ]:
# tensors can also have names (in the computation graph)
named_tensor = tf.constant(7.2, name='my_named_tuple')
named_tensor.numpy()

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-11.png" align="center" width="40%" padding="0px"><br>
    <br>
    Vector (Rank-1) Tensor
</div>

In [ ]:
# float vector tensor
rank_1_tensor = tf.constant([2.0, 3.0, 4.0])
print(rank_1_tensor)

In [ ]:
# Contant Tensors are immutable
try:
    rank_1_tensor.assign(8)
except:
    print('Cannot assign contstants')

<br>

#### **1.3.2 tf.Variable**

___

A [**variable**](https://www.tensorflow.org/guide/variable) is the recommended way to represent data manipulated by your program. Variables are usually weights and biases of a model that are optimized during training, they also indicate the degrees of freedom of the model (what model parameters that can change, thus making the model flexible). The following covers how to create, update, and manage instances of [```tf.variable```](https://www.tensorflow.org/api_docs/python/tf/Variable).

In [ ]:
# Create variable tensor
var = tf.Variable(3.)
var

In [ ]:
# Reassign the value of a Variable
var.assign(4)
var

___

**Remark:** why ```tf.constant()``` yet ```tf.Variable()```?  The lowercase ```c``` in ```tf.constant()``` is intended to indicate that tf.constant is an operation, while the captital ```V``` in ```tf.Variable```, is to indicate that it is a class with many operations.
___

<br>

**Multidimensional Tensors**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-10.png" align="center" width="40%" padding="0px"><br>
    Matrix (Rank-2) Tensor
</div>

In [ ]:
# tensors can be n-arrays, and we can specity data type
rank_2_tensor = tf.constant([[1, 2],
                             [3, 4],
                             [5, 6]], dtype=tf.float16)
print(rank_2_tensor)

<br>

**Build tensors using distributions**

In [ ]:
# we can also create multi dim Variables directly
c = tf.Variable(np.random.randn(3).reshape(3,1)) #reshape
# automatically assings data type
c #

In [ ]:
# Altenatively, we can pass Python Lists or NumPy arrays
my_tensor = tf.constant([[1.0, 2.0], [3.0, 4.0]])
my_var = tf.Variable(my_tensor)
my_var

In [ ]:
# view properties of tensor
print('\nname  : ', my_var.name)
print('\ntype  : ', my_var.dtype)
print('\nshape : ', my_var.shape)
print('\ndevice: ', my_var.device)
print("\nAs NumPy: ", my_var.numpy)

In [ ]:
# Tensor operations
print("\nViewed as a tensor:", tf.convert_to_tensor(my_var))
print("\nIndex of highest value:", tf.argmax(my_var))

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-09.png" align="center" width="50%" padding="0px"><br>
    3D (Rank-3) Tensor
</div>

In [ ]:
# Tensors with 3 or more axes (dimensions)
rank_3_tensor = tf.constant([
  [[0, 1, 2, 3, 4],
   [5, 6, 7, 8, 9]],
  [[10, 11, 12, 13, 14],
   [15, 16, 17, 18, 19]],
  [[20, 21, 22, 23, 24],
   [25, 26, 27, 28, 29]],])
                    
print(rank_3_tensor)

<br>

**Tensor Variable Assignments**

In [ ]:
# This creates a new tensor; it does not reshape my_var.
print("\nCopying and reshaping (this creates a new tensor): ", tf.reshape(my_var, ([1,4])))

In [ ]:
## Sanity Check ##

d = tf.Variable([4.0, 5.5])
# This will keep the same dtype, float32
d.assign([10, 20]) 

# Fails, as it resizes the variable 
try:
  d.assign([1.0, 2.0, 3.0])
except Exception as exc:
  print(f"{type(exc).__name__}: {exc}")

___

**Note:** Calling ```assign( )``` does not, usually, allocate a new tensor - instead the tensor's memory is reused.

___

In [ ]:
## Sanity Check ##

# Create a new variable based on valued of d
e = tf.Variable(d)
# make assignment
d.assign([10, 20]) 

# a and b are different
print(e.numpy())
print(d.numpy())

In [ ]:
# Inplace increase/decrease variable values

f = tf.Variable(3.)
print('\noriginal value: ', f.numpy())
f.assign(10)
print('\nassigned value: ', f.numpy())
print('\nadd 1:', f.assign_add(1.).numpy())
print('\nsubtract 5:', f.assign_sub(5.).numpy())

# same but for arrays
print("\nIncrease/Decrease values in arrays")
print('original value: ',e.numpy())      
print('\nadd [1,4]: ', e.assign_add([1,4]).numpy())  
print('\nsubtract [3,5]', e.assign_sub([3,5]).numpy())  


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>

</div>
<!-------------------------------------->

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-08.png" align="center" width="50%" padding="0px"><br>
    4D (Rank-4) Tensor
</div>


> **To test your understanding of Tensors, create the above Tensor. For simplicity, make all entries equal to zero and name your tensor as** ```rank_4_tensor``` **.** 


<hr style="border: 2px solid#003262;" />

In [ ]:
# your code here




In [ ]:
## Sanity Check ##
#print("\nType of elements:", rank_4_tensor.dtype)
#print("\nNumber of dimensions:", rank_4_tensor.ndim)
#print("\nShape of tensor:", rank_4_tensor.shape)
#print("\nElements along axis 0 of tensor:", rank_4_tensor.shape[0])
#print("\nElements along the last axis of tensor:", rank_4_tensor.shape[-1])
#print("\nTotal number of elements (3*2*4*5): ", tf.size(rank_4_tensor).numpy())

<!--Navigate back to table of contents-->
<br>
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<br>

#### **1.3.3 (Optional) Lifecycles, naming, and watching**

___

Similar to Python Objects, a ```tf.Variable``` instance has an [**object lifecyle**](https://en.wikipedia.org/wiki/Object_lifetime). For example, when there are no references to a variable it is automatically ignored and deallocated. Variable names are preserved when loading and saving models - however, there is no need to assign them as variables in in models are uniquely named automatically.

In [ ]:
# Altenatively, we can pass Python Lists or NumPy arrays
my_tensor = tf.constant([[1.0, 2.0], [3.0, 4.0]])

In [ ]:
# Create a and b; they have the same value but are backed by different tensors.
g = tf.Variable(my_tensor, name="Mark")

# A new variable with the same name, but different value
h = tf.Variable(my_tensor + 1, name="Mark")      # Note that the scalar add is broadcast (see intro_NUMPY for details)

# These are elementwise-unequal, despite having the same name
print(g == h)

<br>

#### **1.3.4 (Optional) Placing variables and tensors**

___

TensorFlow will always attempt to place tensors and variables on the fastest device compatible with the ```dtype```. However, this too is a process that can be overriden if desired. The following shows an example of such an instance. In particular, a float tensor and variable are placed on a CPU even if a GPU is available - most variables are placed on a GPU if one is available. For more details on placing variables and tensors, see [here](https://www.tensorflow.org/guide/variable).

In [ ]:
with tf.device('CPU:0'):

  # Create some tensors
  i = tf.Variable([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
  j = tf.constant([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
  k = tf.matmul(i, j)

print(k)

___

**Note:** Placing variables is an important aspect of TF. This is because it is possible to set the location of a variable or tensor on one device, yet do the computation on another device. This may introduce delay when devices communicate with one-another, but should you want to have multiple GPU workers yet only one copy of the variables this approach is worth considering. See [**automatic distribution**](https://www.tensorflow.org/guide/autodiff) for details.


___

In [ ]:
with tf.device('CPU:0'):
  l = tf.Variable([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
  m = tf.Variable([[1.0, 2.0, 3.0]])
    
with tf.device('GPU:0'):
  # Element-wise multiply
  n = l * m

print(n)

___

**Note:** [**tf.config.set_soft_device_placement**](https://www.tensorflow.org/api_docs/python/tf/config/set_soft_device_placement) is on by default. If you run the code above on a device witout a GPU, the multiplication will be ran on the CPU.
    
___

<a id='Part_1_4'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.4: TENSORFLOW OPERATIONS
<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-04.png" align="center" width="50%" padding="0px"><br>
</div>

<br>

#### **1.4.1 Basic Arithmetic on Tensors**

___

In [ ]:
a = tf.constant([[1, 2],
                 [3, 4]])
b = tf.constant([[10, 20],
                 [30, 40]]) 

In [ ]:
# Addition
print(tf.add(a, b))

In [ ]:
# Alternatively, element-wise addition using operator overloading
print(a + b)

In [ ]:
# Element-wise multiplication
print(tf.multiply(a, b))

In [ ]:
# Alternatively, element-wise multiplication using operator overloading
print(a * b)

In [ ]:
# Use NumPy values
import numpy as np

ab = np.multiply(a, b)
print(ab)

In [ ]:
# Matrix multiplication
print(tf.matmul(a, b))

In [ ]:
# Alternatively, Matrix multiplication
print(a @ b)

<br>

#### **1.4.2 Operations on Tensors**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-02.png" align="center" width="40%" padding="0px"><br>
</div>

___

<br>

The following represent a few operations on tensors. For a comprehensive list see [here](https://www.tensorflow.org/api_docs/python/tf/Tensor).

In [ ]:
c = tf.constant([[3.0, 7.0], [10.0, 2.0]])

In [ ]:
# Find the largest value
print(tf.reduce_max(c))

In [ ]:
# Find the index of the largest value
print(tf.argmax(c))

In [ ]:
# Compute the softmax
print(tf.nn.softmax(c))

<br>

**Gradients from scrach**

> On the following examples, we are using [**tf.GradientTape( )**](https://www.tensorflow.org/api_docs/python/tf/GradientTape) to compute some example gradients. GradientTape is considered an advanced TensorFlow feature, but also an extremely powerful option for [advanced and custom differentiation](https://www.tensorflow.org/guide/advanced_autodiff).

In [ ]:
mpl.rcParams['figure.figsize'] = (6, 4)

In [ ]:
# create a noisy dataset that resembles y=mx+b+some_noise
def noisy_data(m=0.3, b=.3, n=100):
    x = tf.random.uniform(shape=(n,))
    some_noise = tf.random.normal(shape=(len(x),), stddev=0.01)
    y = m * x + b + some_noise
    return(x, y)

In [ ]:
# get some data
x_train, y_train = noisy_data()

In [ ]:
# visualize the data as scatter
plt.plot(x_train, y_train, 'b.')

In [ ]:
# declare variables
m = tf.Variable(0.)
b = tf.Variable(0.)

In [ ]:
# prediction
def pred_response(x):
    y = m * x + b
    return(y)

In [ ]:
# loss function
def squared_error(y_pred, y_act):
    squaredError = tf.reduce_mean(tf.square(y_pred - y_act))
    return(squaredError)

In [ ]:
# compute loss prior to training
loss = squared_error(pred_response(x_train), y_train)
print("Starting loss: ", loss.numpy())

In [ ]:
# gradient descent from scratch
learning_rate = 0.05
steps = 200

for i in range(steps):
    with tf.GradientTape() as tape:
        predictions = pred_response(x_train)
        loss = squared_error (predictions, y_train)
    
    # compute loss with respect to m and b
    gradients = tape.gradient(loss, [m, b])
    
    m.assign_sub(gradients[0] * learning_rate)
    b.assign_sub(gradients[1] * learning_rate)
   
    
    # track steps
    if i % 10 == 0:
        print("Step number: %d, has loss %f" % (i, loss.numpy()))


print("\nGradient after %d steps: " % steps, gradients)        

<br>

#### **1.4.3 Data Selection: Indexing and Slicing a Tensor**
___

<br>

TensorFlow follows standard Python indexing rules. For details see: [**Indexing Tensors**](https://www.tensorflow.org/guide/tensor#indexing)

**Single-Axis Indexing**

In [ ]:
my_tensor = tf.constant([0, 1, 2, 3, 5, 6, 7, 8, 10, 20, 30, 40])
my_tensor

In [ ]:
# indexing with a sclar removes the dimension
print("First:", my_tensor[0].numpy())
print("Second:", my_tensor[1].numpy())
print("Last:", my_tensor[-1].numpy())

<br>

**Single-Axis Slicing**

In [ ]:
print("Everything:", my_tensor[:].numpy())
print("Before 4:", rank_1_tensor[:4].numpy())
print("From 4 to the end:", rank_1_tensor[4:].numpy())
print("From 2, before 8:", rank_1_tensor[2:8].numpy())
print("Every other item:", rank_1_tensor[::2].numpy())
print("Reversed:", rank_1_tensor[::-1].numpy())

___

**Note:** Integer indexing removes the dimension, while range indexing (i.e. slice) keeps the dimension.
    
___

<br>

**Multi-Axis Indexing**

In [ ]:
# reusing rank-2 tensor from before
print(rank_2_tensor.numpy())

In [ ]:
# Pull out a single value from a 2-rank tensor
print(rank_2_tensor[2, 1].numpy())

In [ ]:
# Get row and column tensors using combination of indexing and slices
print("Second row:", rank_2_tensor[1, :].numpy())
print("Second column:", rank_2_tensor[:, 1].numpy())
print("Last row:", rank_2_tensor[-1, :].numpy())
print("First item in last column:", rank_2_tensor[0, -1].numpy())
print("Skip the first row:")
print(rank_2_tensor[1:, :].numpy(), "\n")

<br>

#### **1.4.4 Manipulating Tensor Shapes**
___


<br>

Performing a tensor [**reshape**](https://www.tensorflow.org/guide/tensor#manipulating_shapes) will work so long as the shape of the new tensor has the same number of elements. 

In [ ]:
# Shape returns a `TensorShape` object that shows the size on each dimension
var_x = tf.Variable(tf.constant([[1], [2], [3]]))
print(var_x.shape)

In [ ]:
# Convert tensor object into a Python list
print(var_x.shape.as_list())

In [ ]:
# Reshape a tensor to a new shape by passing a list
reshaped = tf.reshape(var_x, [1, 3])

# sanity check
print(var_x.shape)
print(reshaped.shape)

<br>

#### **1.4.5 (Optional) Broadcasting Tensors**

___

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-06.png" align="center" width="40%" padding="0px"><br>
    Broadcasted Tensor
</div>

<br>

[**Broadcasting**](https://www.tensorflow.org/guide/tensor#broadcasting) in TensorFlow is borrowed directly from the notion of [**NumPy Array Broadcasting**](https://numpy.org/doc/stable/user/basics.broadcasting.html). The general idea is that, under certain conditions, a smaller tensor can be 'stretched' to fit larger tensors when running combined operations on both.  Following are some examples of this concept.

<br>

In [ ]:
# variable declarations
x = tf.constant([5, 7, 11])
y = tf.constant(3)
z = tf.constant([3, 3, 3])


In [ ]:
# Same results
print(tf.multiply(x, 3))
print(x * y)
print(x * z)

In [ ]:
## To really grasp the idea: without broadcasting ##
x_stretch = tf.constant([[5, 5, 5, 5],
                         [7, 7, 7, 7],
                         [11, 11, 11, 11]])

y_stretch = tf.constant([[1, 2, 3, 4],
                         [1, 2, 3, 4],
                         [1, 2, 3, 4]])

# using operator overloading
result = x_stretch * y_stretch

print(result)

In [ ]:
## To really grasp the idea: with broadcasting ##
# These are the same computations
x = tf.reshape(x,[3,1])
y = tf.range(1, 5)
print(x, "\n")
print(y, "\n")
print(tf.multiply(x, y))

<br>

#### **1.4.6 (Optional) Ragged, String, and Sparse Tensors**
___


<br>

TensorFlow has several ways of dealing with 'abnormal' tensors. In particular:

>[**Ragged Tensors**](https://www.tensorflow.org/guide/tensor#ragged_tensors) - variable number of element along some axis.<br>
>[**String Tensors**](https://www.tensorflow.org/guide/tensor#string_tensors) - atomic dtype that cannot be indexed similar to Python strings.<br>
>[**Sparse Tensors**](https://www.tensorflow.org/guide/tensor#sparse_tensors) - for handling sparse data, like a very wide embedding space.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Write a function `normalize(x)` decorated with `@tf.function` that accepts a rank-1 float tensor and returns it rescaled to the range [0, 1] using the formula `(x - tf.reduce_min(x)) / (tf.reduce_max(x) - tf.reduce_min(x))`. Then use `tf.GradientTape` to compute the gradient of the sum of the normalized output with respect to the input `x`. Verify that the gradient has the same shape as `x`.**

<br>

```python
# starter code
import tensorflow as tf

x = tf.constant([2.0, 5.0, 8.0, 1.0, 9.0])

@tf.function
def normalize(x):
    # your implementation here
    ...

# call normalize and compute gradient
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# your code here

<!--Navigate back to table of contents-->
<br>
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **TENSORFLOW** GRAPHS, **AND** EXECUTIONS

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-07.png" align="center" width="40%" padding="0px"><br>
</div>

<br>


#### CONTENTS:

> [PART 2.1: TENSORFLOW COMPUTATION FUNCTION](#Part_2_1)<br>



<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: TENSORFLOW COMPUTATION FUNCTION
<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-01.png" align="center" width="50%" padding="0px"><br>
</div>

<br>

In TensorFlow (TF) 2.0, you can decorate Python functions using ```tf.function``` to mark them for [**just in time (JIT)**](https://docs.w3cub.com/tensorflow~guide/performance/xla/jit/) compilation. Meaning that TensorFlow runs it as [**single graph**](https://github.com/tensorflow/community/pull/20). This change is done to make TensorFlow more 'Pythonic' - which enables eager execution by default, encourages the encapsulation of graph computations as Python functions, and aligns the 'state' in the TensorFlow runtime with the state in the Python program.  All that to say, **Functions, not sessions in TF 2.0**:

```python
    # TensorFlow 1.X
    outputs = session.run(f(placeholder), feed_dict={placeholder: input})
    # TensorFlow 2.0
    outputs = f(input)
```

Just in case it wasn't clear, [**using graphs directly is deprecated in TF 2.0**](https://www.tensorflow.org/api_docs/python/tf/Graph#using_graphs_directly_deprecated).

<br>

#### **2.1.1 TensorFlow Function - \@tf.function**
___


The [**\@tf.function**](https://www.tensorflow.org/api_docs/python/tf/function) compiles a function into a callable [**TensorFlow graph**](https://www.tensorflow.org/api_docs/python/tf/Graph).  Use the ```tf.function``` to get performant and portable models, but note that ```tf.function``` is not a one-size-fits-all solution for faster computation.  For more on common issues you may enconter when using ```tf.function``` and how to deal with them, see [here](https://www.tensorflow.org/guide/function#basics). Now let's get to some TF functions.

In [ ]:
# example function
@tf.function
def f(x,y):
    return(x ** 2 + y)

In [ ]:
x = tf.constant([2, 4])
y = tf.constant([4, -2])

In [ ]:
f(x,y)

<br>

**\@tf.function Control Flow**

In [ ]:
# tf.function() may use data-dependent control flow 
@tf.function() 
def g(x):
    if tf.reduce_sum(x) > 0:
        return(x * x)
    else: 
        return(-x // 2)

In [ ]:
# function call
g(tf.constant(4.7))

___

**Note:** ```tf.function()``` can handle ```if```, ```for```, ```while```, ```continue```, ```break```, and ```return``` control flow statements.
___

<br>

**\@tf.function with** [**tf.Tensor**](https://www.tensorflow.org/api_docs/python/tf/Tensor) **and** [**tf.Variable**](https://www.tensorflow.org/api_docs/python/tf/Variable) **Objects**

In [ ]:
@tf.function
def h():
  return(x ** 2 - y)

In [ ]:
x = tf.constant([-2, -4])
y = tf.Variable([-4, -2])
h()

<br>

**\@tf.function with side effects, such as** [**tf.print**](https://www.tensorflow.org/api_docs/python/tf/print) **and** [**tf.Variable**](https://www.tensorflow.org/api_docs/python/tf/Variable)

In [ ]:
@tf.function
def f_se(x):
  for i in tf.range(x):
    c.assign_add(i)


In [ ]:
# before side-effects
c = tf.Variable(4)
c

In [ ]:
# function call
f_se(3)
# after side-effects
c

___

**Key Point:** Any Python side-effects - such as printing with ```print()```, appending to a list, etc. - will only happen once as ```func```, in this case named ```f_se()```, is traced. If side-effects are desired in [**tf.function**](https://www.tensorflow.org/api_docs/python/tf/function), then they need to be written as TF ops. More on this below.
___

In [ ]:
l = []

# appends only once when tracing
@tf.function
def f(x):
  for i in x:
    l.append(i + 1)    


In [ ]:
# function call
f(tf.constant([2, 4, 9]))


In [ ]:
# examine side-effects
l

In [ ]:
# appends in tf.function
@tf.function
def f(x):
  temp = tf.TensorArray(dtype=tf.int32, size=0, dynamic_size=True)
  for i in range(len(x)):
    temp = temp.write(i, x[i] + 1)
  return(temp.stack())


In [ ]:
# function call
f(tf.constant([2, 4, 9]))

<br>

**\@tf.function is Polymorphic**

In [ ]:
@tf.function
def g(x):
  return(x + 1)

In [ ]:
# function call with int32
g(tf.constant([2, 4, 9]))

In [ ]:
# function call with floats32
g(tf.constant([2.0, 4.0, 9.0]))

___

**Key Point:** [**tf.function**](https://www.tensorflow.org/api_docs/python/tf/function) can build more than one graph to support different data types or shapes as it encounters them. To obtain an individual graph, use the ```get_concrete_function``` method of the callable created by ```tf.function```. Example follows.
___

In [ ]:
# source function for individual graph
@tf.function
def h(x):
  return(tf.abs(x))

In [ ]:
# Build individual graphs
f1 = h.get_concrete_function(1)
f2 = h.get_concrete_function(2)  # Slow 
print("f1 == f2?", f1 is f2)
print(f1)
print(f2)


In [ ]:
# Reuse graph created when f1 is traced
f1 = h.get_concrete_function(tf.constant(1))
f2 = h.get_concrete_function(tf.constant(2))  # Fast 
print("f1 == f2?",f1 is f2)
print(f1)
print(f2)

<br>

**\@tf.function with** [**Input Signatures**](https://www.python.org/dev/peps/pep-0362/)

In [ ]:
@tf.function
def f(x):
  return(x + 1)

In [ ]:
# different shapes
vector = tf.constant([1.0, 1.0])
matrix = tf.constant([[3.0]])

In [ ]:
f1 = f.get_concrete_function(vector) 
f2 = f.get_concrete_function(matrix)

print("f1 == f2?", f1 is f2)
print(f1)
print(f2)

___

**Key Point:** Recall that [**tf.function**](https://www.tensorflow.org/api_docs/python/tf/function) is polymorphic. As such, ```tf.function``` instantiates a separte graph for every unique input - shape or datatypes. An [**input signature**](https://www.python.org/dev/peps/pep-0362/) can be provided to ```tf.function``` to control the graphs traced. This is useful to avoid creating multiple graphs when tensors have dynamic shapes, or to restrict the shape and datatype of tensors that can be used. Example follows.
___

In [ ]:
# modify previous function to include an input signature
@tf.function(
    input_signature=[tf.TensorSpec(shape=None, dtype=tf.float32)])
def g(x):
  return(x + 1)

In [ ]:
# same shapes as before
vector = tf.constant([1.0, 1.0])
matrix = tf.constant([[3.0]])


In [ ]:
f1 = g.get_concrete_function(vector) 
f2 = g.get_concrete_function(matrix)

print("f1 == f2?", f1 is f2)
print(f1)
print(f2)

<br>

___

**Final Comments:** When using TF 2.0, it is recommended that users refactor their code into smaller functions that are called as needed. Moreover, it's not necessary to decorate each of these smaller functions with ```tf.function```; only use ```tf.function``` to decorate high-level computations - for example, one step of training or the forward pass of your model.

Also, when **creating variables** - such as those created locally and returned - keep in mind that a variable can only be created once. If you plan on creating local variables that are returned it is recommended to create stateful objecets like [**tf.Variable**](https://www.tensorflow.org/api_docs/python/tf/Variable) outside of [**tf.function**](https://www.tensorflow.org/api_docs/python/tf/function) and passing them as arguments.
___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Write a `@tf.function`-decorated function called `l2_norm` that accepts a single tensor and returns its L2 norm (the square root of the sum of squared elements). Call it with both an integer tensor and a float tensor to observe polymorphic tracing. Then use `get_concrete_function` to retrieve the concrete graph for float32 inputs and confirm that the float32 graph is reused for a second float32 call.**

<br>

```python
# starter code
import tensorflow as tf

@tf.function
def l2_norm(x):
    # your code here
    ...

int_input   = tf.constant([3, 4])
float_input = tf.constant([3.0, 4.0])

# call with both dtypes, then retrieve the concrete function
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# your code here


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **WRAP-UP** AND **NEXT** STEPS

<br>

You have completed the introduction to TensorFlow V.2. You now have a foundation in tensors, operations, computation graphs, and the `@tf.function` decorator. To continue building with TensorFlow, explore the following topics:

> [TensorFlow Tutorials](https://www.tensorflow.org/tutorials) - Official hands-on guides for common deep learning tasks. <br>
> [Keras Documentation](https://keras.io/) - High-level API for building and training deep learning models. <br>
> [Advanced Autodiff Guide](https://www.tensorflow.org/guide/advanced_autodiff) - Deep dive into gradient tapes and custom differentiation. <br>

<br>

<br>

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Appendix_1'></a>

<hr style="border: 6px solid#003262;" />

#### APPENDIX I


## **TENSORFLOW** INSTALLATION



<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tf_logo_social.png" align="center" width="30%" padding="0px"><br>
    <br>
</div>

<br>

#### **Appendix I.1 Install Python Development Environment**

___

#### **Install/Update pip3**

https://pip.pypa.io/en/stable/installing/

```bash
    # TensorFlow requires pip version >= 19.0
    $ pip install --upgrade pip 
```

<br>

#### **Install/Update Python 3**

https://www.python.org/downloads/

```bash
    # TensorFlow requires Python 3.5-3.8 
    $ sudo apt-get update && sudo apt-get install python3-dev python3-pip python3-venv python-virtualenv
```

<br>

#### **Appendix I.2 Create Virtual Environgment (Recommended)**


___

**Create virtual environment**

>On a terminal, or using magic keys, create a new virtual environment using a Python interpreter and creating a ```venv``` directory to hold it

```bash
    $ python3 -m venv venv
```

<br>

**Activate the virtual environment**

```bash
    $ source ./venv/bin/activate      # sh, bash, or zsh

    $ . ./venv/bin/activate.fish       # fish

    $ source ./venv/bin/activate.csh  # csh or tcsh
```

<br>

**Install packages within virtual environment**

When the virtual environment is active, your shell prompt is prefixed by ```(venv)```

> Step 1: reapeat 1.1.1 above, within ```(venv)```<br>

```bash
    # TensorFlow requires pip version >= 19.0
    $ pip install --upgrade pip 
```    

> Step 2: Install TensorFlow2

```bash
    (venv) $ pip install --upgrade tensorflow
```


> Step 3: Verify install

```bash
    (venv) $ python -c "import tensorflow as tf;print(tf.reduce_sum(tf.random.normal([1000, 1000])))"
```

> Step 4: To exit virtual enviroment later

```bash
    (venv) $ pip deactivate
```

<br>

#### **Appendix I.3 Install TensorFlow on Local Machine (Optional Setup)**

___


**TensorFlow 2**

https://www.tensorflow.org/install/pip

```bash
    # Install using pip
    $ pip install --upgrade tensorflow 
```

<br>

**Verify Installation**

```bash
    $ python -c "import tensorflow as tf;print(tf.reduce_sum(tf.random.normal([1000, 1000])))"
```

<br>

#### **Appendix I.4 Run TensorFlow in a Container (Optional Setup)**

___

<br>

[**Tensorflow Docker images**](https://www.tensorflow.org/install/docker) come pre-configured to run TensorFlow, and provide a virtual environment that is generally the easiest way to set up [**GPU processing**](https://www.tensorflow.org/install/gpu). To learn more about how to use [**Docker**](https://www.tensorflow.org/install/docker/) to separate your applications from your infrastructure, click [here](https://docs.docker.com/get-docker/).

<br>

<a id='Appendix_2'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX II


## **REFERENCES** AND ADDITIONAL **RESOURCES**

<br>


> [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)<br>
> [TensorFlow Guide](https://www.tensorflow.org/guide) <br>
> [Deep Learning with Python by Francois Chollet](https://www.manning.com/books/deep-learning-with-python)
> [Introduction to TensorFlow by Andrew Ng and Kian Katanforoosh](https://cs230.stanford.edu/blog/tensorflow/)<br>
> [Deep Learning Illustrated by Jon Krohn, Grant Beyleveld, and Aglae Bassens](https://www.deeplearningillustrated.com/)<br>
> [Hands-On Introduction to TensorFlow 2.0 by Josh Gordon and Amit Patankar](https://youtu.be/Yyv-ng0_OTU)<br>
> [Getting Started with TensorFlow and Deep Learning by Josh Gordon](https://youtu.be/tYYVSEHq-io)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 6px solid#003262;" />
